# Lesson 10: AI that paints — diffusion models 🎨
**Learning goals**
1. Understand diffusion: **learn to remove noise → "pull" a picture out of pure noise**
2. Add noise to a picture yourself, and try to remove it
3. Paint with a real AI and experiment with prompts and random seeds

**⏱️ Timing:** adding noise 15' | removing noise 10' | AI painting 25' | sharing 10'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🪟 An analogy first
Imagine a foggy window with a cat behind it.
- **Adding noise:** keep breathing on the glass. The cat gets blurrier until all you see is fog.
- **Removing noise:** wipe the fog away bit by bit, and the cat slowly appears.

During training, a diffusion model watches **hundreds of millions of pictures getting "fogged up"** and learns how to wipe.
When it paints, it starts from **pure noise** and wipes step by step — and your prompt tells it whether to wipe "towards a cat" or "towards a castle".

> 🗿 Michelangelo supposedly said: "The statue is already inside the stone; I just remove what isn't needed." Diffusion models work the same way!

## Part 1: Add noise to a picture 🌫️

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

img = data.astronaut() / 255.0          # a photo of an astronaut, values 0–1
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, level in zip(axes, [0, 0.1, 0.25, 0.5, 0.75, 1.0]):
    noise = np.random.randn(*img.shape)
    noisy = np.sqrt(1 - level) * img + np.sqrt(level) * noise   # the same kind of "mixing" diffusion models use
    ax.imshow(np.clip(noisy, 0, 1)); ax.set_title(f"noise {level:.0%}"); ax.axis("off")
plt.show()

### 🧩 Exercise 1
1. Up to what noise level can you **still recognise** the astronaut? When does it disappear completely?
2. Is the last picture (100% noise) any different from any other random noise? That's the **starting point** of AI painting.

## Part 2: Let's try removing noise ourselves 🧽
The simplest way to remove noise: blur the picture (each pixel becomes the average of its neighbours). How well does it work?

In [ ]:
from scipy.ndimage import gaussian_filter

noisy = np.clip(np.sqrt(0.7) * img + np.sqrt(0.3) * np.random.randn(*img.shape), 0, 1)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(noisy); axes[0].set_title("noisy")
for ax, s in zip(axes[1:], [1, 2, 4]):
    ax.imshow(np.clip(gaussian_filter(noisy, sigma=(s, s, 0)), 0, 1)); ax.set_title(f"blur {s}")
for ax in axes: ax.axis("off")
plt.show()

Less noise — but the picture is blurry too! The simple method **can't tell noise from detail**.
The clever thing about AI: it has **seen so many faces** that it knows what an eye should look like, so it can put the detail back. (Remember the denoising autoencoder from Lesson 9? A diffusion model is a much bigger version of that idea, applied many small steps at a time.)

## Part 3: A real AI painter 🖌️

In [ ]:
# ⏳ Download the image model (about 2 minutes, needs a GPU)
!pip -q install diffusers
import torch
from diffusers import AutoPipelineForText2Image

painter = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")

def draw(prompt, seed=0, steps=1):
    g = torch.Generator("cuda").manual_seed(seed)
    return painter(prompt, num_inference_steps=steps, guidance_scale=0.0, generator=g).images[0]

print("✅ The painter is ready")

In [ ]:
draw("a cute robot reading a book in a library, cartoon style")

### 🧩 Exercise 2: Prompt experiments
Everyone tries 3 prompts, changing **only one thing** each time:
1. Change the **subject**: `a cat` → `a dragon`
2. Change the **style**: `cartoon style` / `oil painting` / `pixel art` / `watercolor`
3. Change the **details**: add `at sunset`, `in the snow`, `wearing a hat`

⚠️ Class rule: only draw things you'd be happy to show at school.

In [ ]:
draw("a dragon eating noodles, pixel art")

### 🧩 Exercise 3: The random seed = the starting fog
Same prompt, different `seed` → it starts wiping from **different noise**, so the picture is different.

In [ ]:
prompt = "a lighthouse on a cliff, watercolor"
imgs = [draw(prompt, seed=s) for s in range(4)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, s in zip(axes, imgs, range(4)):
    ax.imshow(im); ax.set_title(f"seed {s}"); ax.axis("off")
plt.show()

### 🚀 Challenge: how many wipes?
`steps` is the number of denoising steps. This model (Turbo) was specially trained to make a picture in just 1 step; try 1, 2 and 4 steps.
**Think:** why does the same seed + same prompt give exactly the same picture every time? What could that mean for proving a picture was made by AI?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, st in zip(axes, [1, 2, 4]):
    ax.imshow(draw("a red sports car, photo", seed=1, steps=st)); ax.set_title(f"{st} steps"); ax.axis("off")
plt.show()

## 📌 Summary
- Training: learn to **wipe away noise**. Painting: wipe a picture out of **pure noise**, step by step
- The prompt decides "which direction to wipe", the seed decides "which fog to start from"
- Next lesson: another way to generate pictures — two AIs **competing** against each other